# Neuronpedia NLA API — Demo Notebook

End-to-end walkthrough of the three NLA endpoints exposed via Neuronpedia's HTTP API:

- `GET  /api/nla/sources` — list available `(modelId, nlaSourceId)` pairs
- `POST /api/nla/completion` — run a chat completion and get canonical NLA tokens back
- `POST /api/nla/explain` — explain NLA activations at specific token positions

Each step below is runnable as-is. The notebook only needs the `requests` library (`pip install requests`).

**Anonymous access is fine** — the API doesn't require a key. With a key from your [account page](https://www.neuronpedia.org/account) you get higher rate limits; set it as an env var named `NEURONPEDIA_API_KEY` to use it.


## Setup


In [18]:
import json
import os

import requests

BASE_URL = "https://www.neuronpedia.org"

# Optional — only set this if you want the higher rate-limit bucket. Anonymous
# requests work fine for everything in this notebook.
API_KEY = os.environ.get("NEURONPEDIA_API_KEY")
HEADERS = {"Content-Type": "application/json"}
if API_KEY:
    HEADERS["x-api-key"] = API_KEY


def pp(obj):
    """Pretty-print a JSON-serialisable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


def check(resp, print_body=True):
    """Print the response JSON; raise on non-2xx so failed cells stop the chain."""
    try:
        body = resp.json()
    except ValueError:
        body = {"_raw": resp.text}
    if print_body:
        pp({"status": resp.status_code, "body": body})
    resp.raise_for_status()
    return body


## 1. List available NLA sources

Each row is one `(modelId, nlaSourceId)` pair you can use in `/explain` and `/completion`. `model.openRouterAvailable: true` means the model can be used with `/completion` (which runs the generation via OpenRouter).


In [ ]:
resp = requests.get(f"{BASE_URL}/api/nla/sources", headers=HEADERS)
sources_body = check(resp, False)
# Print a table of available NLA sources with just modelId and id for each source
sources = []
for source in sources_body.get("sources", []):
    print(f"modelId: {source.get('modelId')}, sourceId: {source.get('id')}")
    sources.append((source.get("modelId"), source.get("id")))

modelId: gemma-3-27b-it, sourceId: kitft-l41
modelId: llama3.3-70b-it, sourceId: kitft-l53


Pick the model + source you want to use. Edit the values below if you want something different.


In [ ]:
# this picks the first source returned
source = sources[0]
MODEL_ID = source[0]
SOURCE_ID = source[1]

print(f"Using ({MODEL_ID}, {SOURCE_ID}).")

matching = [
    s
    for s in sources_body["sources"]
    if s["modelId"] == MODEL_ID and s["id"] == SOURCE_ID
]
assert matching, (
    f"No source named ({MODEL_ID}, {SOURCE_ID}). Pick one from the previous cell."
)
print(f"Using ({MODEL_ID}, {SOURCE_ID}).")
print(
    f"OpenRouter available for completions: {matching[0]['model']['openRouterAvailable']}"
)


Using (gemma-3-27b-it, kitft-l41).
Using (gemma-3-27b-it, kitft-l41).
OpenRouter available for completions: True


## 2. Run a chat completion

`/api/nla/completion` runs the assistant turn via OpenRouter, then tokenizes the _full chat_ with the NLA tokenizer so the returned `tokens[*].position` values are valid inputs to `/api/nla/explain`.

Response shape:

- `completion`: assistant text only
- `full_text`: chat-templated prompt + assistant turn (pass this as `text` to `/explain`)
- `tokens`: NLA tokens covering `full_text` with `{token, token_id, position}`


In [20]:
resp = requests.post(
    f"{BASE_URL}/api/nla/completion",
    headers=HEADERS,
    json={
        "modelId": MODEL_ID,
        "nlaSourceId": SOURCE_ID,
        "messages": [{"role": "user", "content": "What is the capital of France?"}],
        "completion_tokens": 32,
        "temperature": 0.7,
    },
)
completion_body = check(resp)


{
  "status": 200,
  "body": {
    "completion": "The capital of France is **Paris**. \n\nIt's known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral,",
    "full_text": "<start_of_turn>user\nWhat is the capital of France?<end_of_turn>\n<start_of_turn>model\nThe capital of France is **Paris**. \n\nIt's known for its iconic landmarks like the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral,",
    "tokens": [
      {
        "token": "<bos>",
        "token_id": 2,
        "position": 0,
        "fragment_index": 0,
        "fragment_count": 1
      },
      {
        "token": "<start_of_turn>",
        "token_id": 105,
        "position": 1,
        "fragment_index": 0,
        "fragment_count": 1
      },
      {
        "token": "user",
        "token_id": 2364,
        "position": 2,
        "fragment_index": 0,
        "fragment_count": 1
      },
      {
        "token": "\n",
        "token_id": 107,
        "position": 3,
        "frag

## 3. Explain positions inside the completion

Pick a few `position` values from the previous cell's `tokens` (typically positions inside the assistant turn) and feed them to `/api/nla/explain` along with the `full_text` you got back.


In [21]:
full_text = completion_body["full_text"]
all_tokens = completion_body["tokens"]

# Print the token list with positions so it's easy to pick interesting ones.
for t in all_tokens:
    print(f"  position={t['position']:>3}  token={t['token']!r}")


  position=  0  token='<bos>'
  position=  1  token='<start_of_turn>'
  position=  2  token='user'
  position=  3  token='\n'
  position=  4  token='What'
  position=  5  token=' is'
  position=  6  token=' the'
  position=  7  token=' capital'
  position=  8  token=' of'
  position=  9  token=' France'
  position= 10  token='?'
  position= 11  token='<end_of_turn>'
  position= 12  token='\n'
  position= 13  token='<start_of_turn>'
  position= 14  token='model'
  position= 15  token='\n'
  position= 16  token='The'
  position= 17  token=' capital'
  position= 18  token=' of'
  position= 19  token=' France'
  position= 20  token=' is'
  position= 21  token=' **'
  position= 22  token='Paris'
  position= 23  token='**.'
  position= 24  token=' '
  position= 25  token='\n\n'
  position= 26  token='It'
  position= 27  token="'"
  position= 28  token='s'
  position= 29  token=' known'
  position= 30  token=' for'
  position= 31  token=' its'
  position= 32  token=' iconic'
  position= 33  t

In [22]:
# Edit `positions` to whatever you want to study — max 16 new positions per call.
positions_to_explain = [
    p for p in (len(all_tokens) - 5, len(all_tokens) - 3, len(all_tokens) - 2) if p >= 0
]

resp = requests.post(
    f"{BASE_URL}/api/nla/explain",
    headers=HEADERS,
    json={
        "modelId": MODEL_ID,
        "nlaSourceId": SOURCE_ID,
        "text": full_text,
        "positions": positions_to_explain,
        "temperature": 0.7,
    },
)
check(resp)


{
  "status": 200,
  "body": {
    "results": [
      {
        "token": " Notre",
        "token_id": 42979,
        "position": 43,
        "l2_norm": 43857.265625,
        "description": "Structured FAQ/listicle format established: a formatted article with sections covering global history, covering AI and travel topics.\n\nThe sentence \"Paris, a vibrant capital, is home to many attractions including the Louvre Museum, Eiffel Tower, and Notre\" signals a list of famous landmarks, establishing a list-enumeration pattern of tourist destinations.\n\nFinal token \"Notre\" begins a well-known Parisian landmark name (\"Notre\"), immediately requiring a noun like \"Dame Cathedral\" or \"Notre Dame Cathedral\" to complete the iconic landmark reference. or \"Cathedral\" or \"Dame Cathedral, Paris.\" or \"Dame Cathedral.\" or \"Dame Cathedral.\" or \"Notre Dame.\" or \"Cathedral.\" or.\"",
        "mse": 0.011645372956991196,
        "cosine_similarity": 0.9941765666007996,
        "generated

{'results': [{'token': ' Notre',
   'token_id': 42979,
   'position': 43,
   'l2_norm': 43857.265625,
   'description': 'Structured FAQ/listicle format established: a formatted article with sections covering global history, covering AI and travel topics.\n\nThe sentence "Paris, a vibrant capital, is home to many attractions including the Louvre Museum, Eiffel Tower, and Notre" signals a list of famous landmarks, establishing a list-enumeration pattern of tourist destinations.\n\nFinal token "Notre" begins a well-known Parisian landmark name ("Notre"), immediately requiring a noun like "Dame Cathedral" or "Notre Dame Cathedral" to complete the iconic landmark reference. or "Cathedral" or "Dame Cathedral, Paris." or "Dame Cathedral." or "Dame Cathedral." or "Notre Dame." or "Cathedral." or."',
   'mse': 0.011645372956991196,
   'cosine_similarity': 0.9941765666007996,
   'generated': False,
   'fragment_index': 0,
   'fragment_count': 1},
  {'token': 'Dame',
   'token_id': 138835,
   'po

## 4. Explain a prompt without running a completion

If you only want to study what's happening inside a prompt (no assistant generation), pass `messages` directly to `/api/nla/explain` and the server will apply the model's chat template before tokenizing.


In [23]:
resp = requests.post(
    f"{BASE_URL}/api/nla/explain",
    headers=HEADERS,
    json={
        "modelId": MODEL_ID,
        "nlaSourceId": SOURCE_ID,
        "messages": [{"role": "user", "content": "What is the capital of France?"}],
        "positions": [4, 5, 6],
    },
)
check(resp)


{
  "status": 200,
  "body": {
    "results": [
      {
        "token": "What",
        "token_id": 3689,
        "position": 4,
        "l2_norm": 52710.66015625,
        "description": "AI/chatbot-style query response pattern: a conversational or technical answer about a topic, likely a writing or business question.\n\nThe phrase \"How do you...\" signals a question or request for advice/information, likely a listicle or forum post about a topic.\n\nFinal token \"What\" opens a new question or prompt (\"What\"), immediately expecting a question or topic request, likely a specific topic like \"are the differences between...\" or \"is the best way to...\" or a question mark/entry to begin a question about the Beatles or programming. or \"are some examples of...\" or \"role/advice did...\" or \"are the pros and cons of...\" or \"programming.\"",
        "mse": 0.013030859641730785,
        "cosine_similarity": 0.9934845566749573,
        "generated": false,
        "fragment_index": 0,

{'results': [{'token': 'What',
   'token_id': 3689,
   'position': 4,
   'l2_norm': 52710.66015625,
   'description': 'AI/chatbot-style query response pattern: a conversational or technical answer about a topic, likely a writing or business question.\n\nThe phrase "How do you..." signals a question or request for advice/information, likely a listicle or forum post about a topic.\n\nFinal token "What" opens a new question or prompt ("What"), immediately expecting a question or topic request, likely a specific topic like "are the differences between..." or "is the best way to..." or a question mark/entry to begin a question about the Beatles or programming. or "are some examples of..." or "role/advice did..." or "are the pros and cons of..." or "programming."',
   'mse': 0.013030859641730785,
   'cosine_similarity': 0.9934845566749573,
   'generated': False,
   'fragment_index': 0,
   'fragment_count': 1},
  {'token': ' is',
   'token_id': 563,
   'position': 5,
   'l2_norm': 53924.30468